In [33]:
from langchain_community.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
import requests
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage , ToolMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [21]:
load_dotenv()

True

In [22]:
# tool create

@tool
def get_conversion_factor(base_currency : str , target_currency : str) -> float:
    """ 
    This function fetches the currency conversion factor between a given base currency and a target currency.
    
    """
    url = f'https://v6.exchangerate-api.com/v6/a6a5968f363199c5aba4a74d/pair/{base_currency}/{target_currency}'

    response = requests.get(url)

    return response.json()




@tool
def convert(base_currency_value: int , conversion_rate : Annotated[float,InjectedToolArg]) -> float:
    """
    Given a currency conversion rate this function calculates the target currency value from a given base currency value.
    """

    return base_currency_value * conversion_rate
    


In [23]:
get_conversion_factor.invoke({'base_currency' : 'USD' , 'target_currency' : 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1773273601,
 'time_last_update_utc': 'Thu, 12 Mar 2026 00:00:01 +0000',
 'time_next_update_unix': 1773360001,
 'time_next_update_utc': 'Fri, 13 Mar 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 92.2032}

In [24]:
convert.invoke({'base_currency_value' : 10 , 'conversion_rate' : 92.2})

922.0

## tool binding

In [25]:
llm = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash',
    temperature = 0
)

In [26]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

## Tool Calling

In [27]:
messages = [HumanMessage('what is the conversion factor between USD and INR , and based on that can you convert 10 usd to inr ?')]

In [28]:
messages

[HumanMessage(content='what is the conversion factor between USD and INR , and based on that can you convert 10 usd to inr ?', additional_kwargs={}, response_metadata={})]

In [29]:
ai_message = llm_with_tools.invoke(messages)

In [34]:
messages.append(ai_message)

In [31]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'bed01d8a-6cb7-4d8f-b3f9-af8026043b06',
  'type': 'tool_call'}]

In [35]:
for tool_call in ai_message.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
        # Execute get_conversion_factor and get the result
        result = get_conversion_factor.invoke(tool_call['args'])
        conversion_rate = result['conversion_rate']  # extract rate from API response

        # Append ToolMessage so LLM knows the tool result
        messages.append(ToolMessage(
            content=str(result),
            tool_call_id=tool_call['id']
        ))

In [36]:
ai_message_2 = llm_with_tools.invoke(messages)
messages.append(ai_message_2)


In [37]:
for tool_call in ai_message_2.tool_calls:
    if tool_call['name'] == 'convert':
        # Inject conversion_rate at runtime — LLM only provides base_currency_value
        result = convert.invoke({
            **tool_call['args'],
            'conversion_rate': conversion_rate  # <-- injected here
        })

        messages.append(ToolMessage(
            content=str(result),
            tool_call_id=tool_call['id']
        ))

In [38]:
final_response = llm_with_tools.invoke(messages)
print(final_response.content)

[{'type': 'text', 'text': 'The conversion factor between USD and INR is 92.2032.\n10 USD is equal to 922.03 INR.', 'extras': {'signature': 'CvcBAb4+9vtGFUuhrFML/g3LyAMiTAyQg3ebAZoH8yxs/uZZSeJjJxoIXJ1tGqXy1mkH1iMyitGR9deuAeYzr7sy72Lq5guF0Ov8u3ERRTq1+K3KffYXiRt84s6UThXA6a/d7tboSgR30HGezlrOJ+yfBTf/lejbcMau9lp4DLbHzF/9TomPlmnf/XPtiRE3yVI3nGQi6mTCvTxM/wHcXDx4tWpLqCnL7fUfk+OvSv/oRtMHCD3qMSB44m7f1cuOLpgdVKqfheu9W54P2qVkLXoix7sTiR0z36yoad+prxXGlVLFGIIwwVc6eccg8ZVD6fAwvBjpP90PaA=='}}]
